In [1]:
print(123)

123


In [2]:
from starter import rag

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

It keeps calling the model in a `while True` loop and checks whether the response contains any `function_call` items.

- If there is a function call, it runs the tool, appends the tool result to `messages`, and loops again.
- If there are no function calls, it breaks out of the loop and stops.

So the stop condition is simply: **no function calls in the model’s response**.


In [3]:
"""
OTel - ConsoleSpanExporter
"""

from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

"""
With the tracer in hand, you can wrap any block of code in a span:

with tracer.start_as_current_span("my_operation") as span:
    # your code here
    span.set_attribute("my_key", "my_value")
"""

'\nWith the tracer in hand, you can wrap any block of code in a span:\n\nwith tracer.start_as_current_span("my_operation") as span:\n    # your code here\n    span.set_attribute("my_key", "my_value")\n'

In [4]:
"""
Q1. First trace
Wrap the rag() method so each call produces a span. The simplest way is to create a RAGTraced subclass of RAGBase that wraps rag(), search(), and llm() each in their own span.

Run this query:

How does the agentic loop keep calling the model until it stops?

The console exporter prints every finished span as a dictionary. Count the spans in the console output - each one is a separate ReadableSpan entry. How many spans does the trace produce?

1
3
5
7
"""

'\nQ1. First trace\nWrap the rag() method so each call produces a span. The simplest way is to create a RAGTraced subclass of RAGBase that wraps rag(), search(), and llm() each in their own span.\n\nRun this query:\n\nHow does the agentic loop keep calling the model until it stops?\n\nThe console exporter prints every finished span as a dictionary. Count the spans in the console output - each one is a separate ReadableSpan entry. How many spans does the trace produce?\n\n1\n3\n5\n7\n'

In [5]:
# RAGTraced

from rag_helper import RAGBase

class RAGTraced(RAGBase):

    # def __init__(self, *args, **kwargs):
    #     super().__init__(*args, **kwargs)

    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search_operation") as span:
            return self.index.search(query, num_results=num_results)
            # span.set_attribute("my_key", "my_value")
    
    def llm(self, prompt):
        with tracer.start_as_current_span("llm_operation") as span:
            input_messages = [
                {'role': 'developer', 'content': self.instructions},
                {'role': 'user', 'content': prompt}
            ]

            response = self.llm_client.responses.create(
                model=self.model,
                input=input_messages
            )

            return response
            # span.set_attribute("my_key", "my_value")
        

    def rag(self, query):
        with tracer.start_as_current_span("rag_operation") as span:
            search_results = self.search(query)
            prompt = self.build_prompt(query, search_results)
            response = self.llm(prompt)
            return response.output_text
            # span.set_attribute("my_key", "my_value")


In [6]:
with tracer.start_as_current_span("test-span"):
    print("hello")

hello
{
    "name": "test-span",
    "context": {
        "trace_id": "0x9331f6bd59387be0f26f57937459f129",
        "span_id": "0x3b17af382c5d7c8d",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-08-06T18:43:28.344683Z",
    "end_time": "2026-08-06T18:43:28.344774Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "ab10e358-cbfc-4b80-a29b-a2dbde7a07ea",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}


In [7]:
from starter import index, client
rag = RAGTraced(index=index, llm_client=client)

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

{
    "name": "search_operation",
    "context": {
        "trace_id": "0xcc9193cbe25b67cf0066fa2b496923a6",
        "span_id": "0x07c021d9ee8807cd",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x9458fa965a96fe30",
    "start_time": "2026-08-06T18:43:28.373348Z",
    "end_time": "2026-08-06T18:43:28.376186Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "ab10e358-cbfc-4b80-a29b-a2dbde7a07ea",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}


{
    "name": "llm_operation",
    "context": {
        "trace_id": "0xcc9193cbe25b67cf0066fa2b496923a6",
        "span_id": "0x78354a4e5dfb3fff",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x9458fa965a96fe30",
    "start_time": "2026-08-06T18:43:28.379117Z",
    "end_time": "2026-08-06T18:43:30.490010Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "ab10e358-cbfc-4b80-a29b-a2dbde7a07ea",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "rag_operation",
    "context": {
        "trace_id": "0xcc9193cbe25b67cf0066fa2b496923a6",
        "span_id": "0x9458fa965a96fe30",
        "trace_state": "[]"
    },
    

In [11]:
# -> SOLUTION: 3 with 
"""
{
    "name": "search_operation",
    "context": {
        "trace_id": "0xcc9193cbe25b67cf0066fa2b496923a6",
        "span_id": "0x07c021d9ee8807cd",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x9458fa965a96fe30",
    "start_time": "2026-08-06T18:43:28.373348Z",
    "end_time": "2026-08-06T18:43:28.376186Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "ab10e358-cbfc-4b80-a29b-a2dbde7a07ea",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm_operation",
    "context": {
        "trace_id": "0xcc9193cbe25b67cf0066fa2b496923a6",
        "span_id": "0x78354a4e5dfb3fff",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x9458fa965a96fe30",
    "start_time": "2026-08-06T18:43:28.379117Z",
    "end_time": "2026-08-06T18:43:30.490010Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "ab10e358-cbfc-4b80-a29b-a2dbde7a07ea",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "rag_operation",
    "context": {
        "trace_id": "0xcc9193cbe25b67cf0066fa2b496923a6",
        "span_id": "0x9458fa965a96fe30",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-08-06T18:43:28.373297Z",
    "end_time": "2026-08-06T18:43:30.490910Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "ab10e358-cbfc-4b80-a29b-a2dbde7a07ea",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
The loop keeps calling the model by checking whether the response contains any `function_call` items.

- It sends the current `messages` history to the model.
- If the model returns a function call, your code runs the tool, appends the tool output to `messages`, and sets `has_function_calls = True`.
- After processing the response, if `has_function_calls` is still `False`, the loop breaks.
- So the loop stops when the model returns a final message with no more tool calls.

In short: **no function calls this turn means we’re done**.
"""

'\n{\n    "name": "search_operation",\n    "context": {\n        "trace_id": "0xcc9193cbe25b67cf0066fa2b496923a6",\n        "span_id": "0x07c021d9ee8807cd",\n        "trace_state": "[]"\n    },\n    "kind": "SpanKind.INTERNAL",\n    "parent_id": "0x9458fa965a96fe30",\n    "start_time": "2026-08-06T18:43:28.373348Z",\n    "end_time": "2026-08-06T18:43:28.376186Z",\n    "status": {\n        "status_code": "UNSET"\n    },\n    "attributes": {},\n    "events": [],\n    "links": [],\n    "resource": {\n        "attributes": {\n            "telemetry.sdk.language": "python",\n            "telemetry.sdk.name": "opentelemetry",\n            "telemetry.sdk.version": "1.44.0",\n            "service.instance.id": "ab10e358-cbfc-4b80-a29b-a2dbde7a07ea",\n            "service.name": "unknown_service"\n        },\n        "schema_url": ""\n    }\n}\n{\n    "name": "llm_operation",\n    "context": {\n        "trace_id": "0xcc9193cbe25b67cf0066fa2b496923a6",\n        "span_id": "0x78354a4e5dfb3fff",\n

In [12]:
"""
Q2. Capturing metrics as span attributes
Spans are not just timing markers - you can attach any information you want to them with set_attribute. We already use spans to record how long each step takes. Now we'll add the metrics we care about: tokens and cost.

Read the token usage from the LLM response (the llm() method in the starter already returns the raw response object) and set them as attributes on the llm span:

span.set_attribute("input_tokens", usage.input_tokens)
span.set_attribute("output_tokens", usage.output_tokens)
And since we know both input and output tokens, we can also compute the cost using the code from the previous modules.

Now re-run the query. How many input tokens do we see?

700
7000
70000
700000
"""

'\nQ2. Capturing metrics as span attributes\nSpans are not just timing markers - you can attach any information you want to them with set_attribute. We already use spans to record how long each step takes. Now we\'ll add the metrics we care about: tokens and cost.\n\nRead the token usage from the LLM response (the llm() method in the starter already returns the raw response object) and set them as attributes on the llm span:\n\nspan.set_attribute("input_tokens", usage.input_tokens)\nspan.set_attribute("output_tokens", usage.output_tokens)\nAnd since we know both input and output tokens, we can also compute the cost using the code from the previous modules.\n\nNow re-run the query. How many input tokens do we see?\n\n700\n7000\n70000\n700000\n'

In [13]:
# RAGTraced

from rag_helper import RAGBase

class RAGTraced(RAGBase):

    # def __init__(self, *args, **kwargs):
    #     super().__init__(*args, **kwargs)

    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search_operation") as span:
            return self.index.search(query, num_results=num_results)
            # span.set_attribute("my_key", "my_value")
    
    def llm(self, prompt):
        with tracer.start_as_current_span("llm_operation") as span:
            input_messages = [
                {'role': 'developer', 'content': self.instructions},
                {'role': 'user', 'content': prompt}
            ]

            response = self.llm_client.responses.create(
                model=self.model,
                input=input_messages
            )

            usage = response.usage
            span.set_attribute("input_tokens", usage.input_tokens)
            span.set_attribute("output_tokens", usage.output_tokens)
            
            return response

        

    def rag(self, query):
        with tracer.start_as_current_span("rag_operation") as span:
            search_results = self.search(query)
            prompt = self.build_prompt(query, search_results)
            response = self.llm(prompt)
            return response.output_text
            # span.set_attribute("my_key", "my_value")


In [14]:
from starter import index, client
rag = RAGTraced(index=index, llm_client=client)

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

{
    "name": "search_operation",
    "context": {
        "trace_id": "0x303a141da1a3d803352e48f5cfdea8d4",
        "span_id": "0xfc02ca64817943e4",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x08be8720f7c19a4c",
    "start_time": "2026-08-06T19:02:43.099172Z",
    "end_time": "2026-08-06T19:02:43.103395Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "ab10e358-cbfc-4b80-a29b-a2dbde7a07ea",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm_operation",
    "context": {
        "trace_id": "0x303a141da1a3d803352e48f5cfdea8d4",
        "span_id": "0x3349ff77dfac06a7",
        "trace_state": "[]"
    },
 

In [15]:
# SOLUTION: 7000 WITH 

"""
{
    "name": "search_operation",
    "context": {
        "trace_id": "0x303a141da1a3d803352e48f5cfdea8d4",
        "span_id": "0xfc02ca64817943e4",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x08be8720f7c19a4c",
    "start_time": "2026-08-06T19:02:43.099172Z",
    "end_time": "2026-08-06T19:02:43.103395Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "ab10e358-cbfc-4b80-a29b-a2dbde7a07ea",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm_operation",
    "context": {
        "trace_id": "0x303a141da1a3d803352e48f5cfdea8d4",
        "span_id": "0x3349ff77dfac06a7",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x08be8720f7c19a4c",
    "start_time": "2026-08-06T19:02:43.104452Z",
    "end_time": "2026-08-06T19:02:44.883846Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "input_tokens": 7111,
        "output_tokens": 89
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "ab10e358-cbfc-4b80-a29b-a2dbde7a07ea",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "rag_operation",
    "context": {
        "trace_id": "0x303a141da1a3d803352e48f5cfdea8d4",
        "span_id": "0x08be8720f7c19a4c",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-08-06T19:02:43.099125Z",
    "end_time": "2026-08-06T19:02:44.884578Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "ab10e358-cbfc-4b80-a29b-a2dbde7a07ea",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
It keeps calling the model in a `while True` loop and checks whether the model returned any `function_call` items.

- If there is a function call, the code runs the tool, appends the tool output to the message history, and loops again.
- If there are no function calls in that turn, it breaks out of the loop.

So the stopping condition is: **no function calls this iteration**.
"""

'\n{\n    "name": "search_operation",\n    "context": {\n        "trace_id": "0x303a141da1a3d803352e48f5cfdea8d4",\n        "span_id": "0xfc02ca64817943e4",\n        "trace_state": "[]"\n    },\n    "kind": "SpanKind.INTERNAL",\n    "parent_id": "0x08be8720f7c19a4c",\n    "start_time": "2026-08-06T19:02:43.099172Z",\n    "end_time": "2026-08-06T19:02:43.103395Z",\n    "status": {\n        "status_code": "UNSET"\n    },\n    "attributes": {},\n    "events": [],\n    "links": [],\n    "resource": {\n        "attributes": {\n            "telemetry.sdk.language": "python",\n            "telemetry.sdk.name": "opentelemetry",\n            "telemetry.sdk.version": "1.44.0",\n            "service.instance.id": "ab10e358-cbfc-4b80-a29b-a2dbde7a07ea",\n            "service.name": "unknown_service"\n        },\n        "schema_url": ""\n    }\n}\n{\n    "name": "llm_operation",\n    "context": {\n        "trace_id": "0x303a141da1a3d803352e48f5cfdea8d4",\n        "span_id": "0x3349ff77dfac06a7",\n

In [16]:
"""
Q3. Span timing
Each span automatically records its duration. Look at the console output from Q1 and find the durations for the search span and the llm span.

For a typical query, roughly how long does the LLM call take?

Under 100ms
100-500ms
500-2000ms
Over 2000ms
The first call can be slower (cold start). Pick the range you see most often.
"""


# -> SOLUTION:  OVER 2000 MS WITH 
"""
 "start_time": "2026-08-06T18:43:28.379117Z",
    "end_time": "2026-08-06T18:43:30.490010Z",
"""

'\n "start_time": "2026-08-06T18:43:28.379117Z",\n    "end_time": "2026-08-06T18:43:30.490010Z",\n'

In [17]:
"""
Q4. Saving traces to SQLite
Right now the spans are printed to the terminal and then gone. We don't save them.

We want to persist them so we can query them later.

In this homework, we'll use SQLite - it's a more lightweight option than Postgres, so we don't need to set up any docker containers in this homework.

Our instrumentation is already done, we don't need to change anything there. But we need to create a custom exporter. Instead of printing the spans, it will save them to the database.

OTel calls the exporter through the same span processor we already use, we just swap the destination.

Now we will create a custom exporter that saves each finished span to a SQLite database. The exporter extends SpanExporter. It has the following methods:

- export method that receives a list of ReadableSpan objects
- shutdown and force_flush methods


Let's implement it:


"""

"\nQ4. Saving traces to SQLite\nRight now the spans are printed to the terminal and then gone. We don't save them.\n\nWe want to persist them so we can query them later.\n\nIn this homework, we'll use SQLite - it's a more lightweight option than Postgres, so we don't need to set up any docker containers in this homework.\n\nOur instrumentation is already done, we don't need to change anything there. But we need to create a custom exporter. Instead of printing the spans, it will save them to the database.\n\nOTel calls the exporter through the same span processor we already use, we just swap the destination.\n\nNow we will create a custom exporter that saves each finished span to a SQLite database. The exporter extends SpanExporter. It has the following methods:\n\n- export method that receives a list of ReadableSpan objects\n- shutdown and force_flush methods\n\n\nLet's implement it:\n\n\n"

In [18]:
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

In [19]:
provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)

In [ ]:
"""
Re-run the query from Q1. Which span names appear in the spans table?

Only rag
rag and llm
rag, search, and llm
search, llm, and judge
"""

from starter import index, client
rag = RAGTraced(index=index, llm_client=client)

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)


# SOLUTION: rag, search, and llm WITH: 

"""
sqlite> select * from spans;
name              start_time           end_time             input_tokens  output_tokens  cost
----------------  -------------------  -------------------  ------------  -------------  ----
search_operation  1786044043310812189  1786044043313798855                                   
llm_operation     1786044043323718084  1786044050580007292  7111          116                
rag_operation    
"""

{
    "name": "search_operation",
    "context": {
        "trace_id": "0x6f6bbefc62b132e38841823923d8f0ca",
        "span_id": "0x91ef3780a13b8ba7",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x35b325dbf5719fd5",
    "start_time": "2026-08-06T19:20:43.310812Z",
    "end_time": "2026-08-06T19:20:43.313799Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "ab10e358-cbfc-4b80-a29b-a2dbde7a07ea",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm_operation",
    "context": {
        "trace_id": "0x6f6bbefc62b132e38841823923d8f0ca",
        "span_id": "0x43606bae3a931639",
        "trace_state": "[]"
    },
 

In [ ]:
"""
Q5. Querying trace data
The traces are now in SQLite. Run one more query through the traced RAG, then query the database.

The rag span wraps everything, so its duration includes both search and llm. To see where time actually goes, exclude the rag span and compare the children.

Using SQL (or pandas), compute the total duration for each span name excluding rag. Which span type takes the most total time?

search
llm
They're all about the same
"""

In [ ]:

# run one more query

query = "How do I explain to a recruiter what I learned in this course llm zoomcamp?"
answer = rag.rag(query)
print(answer)


{
    "name": "search_operation",
    "context": {
        "trace_id": "0x99faf0e66eb2eb167c369c5c8c2d3312",
        "span_id": "0xbf889fd864ed89fd",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x45ac5a8b4172b0bf",
    "start_time": "2026-08-06T19:29:00.264286Z",
    "end_time": "2026-08-06T19:29:00.268290Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "ab10e358-cbfc-4b80-a29b-a2dbde7a07ea",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm_operation",
    "context": {
        "trace_id": "0x99faf0e66eb2eb167c369c5c8c2d3312",
        "span_id": "0x975cabacfababb9b",
        "trace_state": "[]"
    },
 

'\nsqlite> select * from spans;\nname              start_time           end_time             input_tokens  output_tokens  cost\n----------------  -------------------  -------------------  ------------  -------------  ----\nsearch_operation  1786044043310812189  1786044043313798855                                   \nllm_operation     1786044043323718084  1786044050580007292  7111          116                \nrag_operation    \n'

In [22]:
# query db

import sqlite3

connection = sqlite3.connect("traces.db")
cursor = connection.cursor()

cursor.execute(
    """
    select * 
    from spans
    where name not like 'rag_operation' 
    """
)

# connection.commit()
connection.close()


In [ ]:
# query db

import pandas as pd
import sqlite3

with sqlite3.connect("traces.db") as connection:
    
    query = """
        select 
            name,
            sum(end_time - start_time) as total_time 
        from spans
        where name not like 'rag_operation' 
        group by name
    """

    df = pd.read_sql_query(query, connection)


print(df)

# SOLUTION: llm 


               name  total_time
0     llm_operation  9933107528
1  search_operation     6990574


In [26]:
"""
Q6. Token stability across runs
Load the SQLite data with pandas. One thing a dashboard can tell you is how stable your system is. If the same query always produces the same number of input tokens, the context your RAG retrieves is consistent. If it varies a lot, something in the search may be unstable.

Run the same query from Q1 three more times (so you have 4 RAG calls total in the database). Then compute the input tokens for each llm span.

How much do the input tokens vary across these 4 runs?

They're identical
Within 10% of each other
Within 50% of each other
They vary more than 50%
"""


# query db

import pandas as pd
import sqlite3

with sqlite3.connect("traces.db") as connection:
    
    query = """
        select 
            * 
        from spans
        -- where name not like 'rag_operation' 
    """

    df = pd.read_sql_query(query, connection)


df

# SOLUTION: llm 


,name,start_time,end_time,input_tokens,output_tokens,cost
0,search_operation,1786044043310812189,1786044043313798855,NaN,NaN,None
1,llm_operation,1786044043323718084,1786044050580007292,7111.0,116.0,None
2,rag_operation,1786044043310752230,1786044050594811450,NaN,NaN,None
3,search_operation,1786044540264286155,1786044540268290063,NaN,NaN,None
4,llm_operation,1786044540282141184,1786044542958959504,5354.0,184.0,None
5,rag_operation,1786044540264223352,1786044542964641284,NaN,NaN,None


In [28]:
with sqlite3.connect("traces.db") as connection:
    cursor = connection.cursor()
    cursor.execute("create table spans_bkp as select * from spans;")
    
connection.close()



In [34]:
with sqlite3.connect("traces.db") as connection:
    df = pd.read_sql_query("select * from spans_bkp", connection)
    
df



,name,start_time,end_time,input_tokens,output_tokens,cost
0,search_operation,1786044043310812189,1786044043313798855,NaN,NaN,None
1,llm_operation,1786044043323718084,1786044050580007292,7111.0,116.0,None
2,rag_operation,1786044043310752230,1786044050594811450,NaN,NaN,None
3,search_operation,1786044540264286155,1786044540268290063,NaN,NaN,None
4,llm_operation,1786044540282141184,1786044542958959504,5354.0,184.0,None
5,rag_operation,1786044540264223352,1786044542964641284,NaN,NaN,None


In [ ]:
with sqlite3.connect("traces.db") as connection:
    cursor = connection.cursor()
    cursor.execute("delete from spans;")
    
connection.close()


In [35]:
with sqlite3.connect("traces.db") as connection:
    df = pd.read_sql_query("select * from spans", connection)
    
df



,name,start_time,end_time,input_tokens,output_tokens,cost


In [36]:
# run query 1 four times: 

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

{
    "name": "search_operation",
    "context": {
        "trace_id": "0xf89f48acd2f744a5551cee752f8f2c1b",
        "span_id": "0x9e3d9caac9d91acf",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xaab0dd881f046a2b",
    "start_time": "2026-08-06T20:04:40.882456Z",
    "end_time": "2026-08-06T20:04:40.887628Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "ab10e358-cbfc-4b80-a29b-a2dbde7a07ea",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm_operation",
    "context": {
        "trace_id": "0xf89f48acd2f744a5551cee752f8f2c1b",
        "span_id": "0x6afb6ec1849330a2",
        "trace_state": "[]"
    },
 

In [37]:
query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

{
    "name": "search_operation",
    "context": {
        "trace_id": "0x6849ca6eb7261ac0a3f2123ccfe6227a",
        "span_id": "0x5d50e27055baa118",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x52542a6ffe642c1c",
    "start_time": "2026-08-06T20:04:57.254249Z",
    "end_time": "2026-08-06T20:04:57.262853Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "ab10e358-cbfc-4b80-a29b-a2dbde7a07ea",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm_operation",
    "context": {
        "trace_id": "0x6849ca6eb7261ac0a3f2123ccfe6227a",
        "span_id": "0xb4a55b79ef0bbd2f",
        "trace_state": "[]"
    },
 

In [38]:
query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

{
    "name": "search_operation",
    "context": {
        "trace_id": "0x2426bf4171a46efc71d797573eb8640e",
        "span_id": "0x603017d5620b91ae",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xf08387add2426297",
    "start_time": "2026-08-06T20:05:00.961474Z",
    "end_time": "2026-08-06T20:05:00.965587Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "ab10e358-cbfc-4b80-a29b-a2dbde7a07ea",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm_operation",
    "context": {
        "trace_id": "0x2426bf4171a46efc71d797573eb8640e",
        "span_id": "0x3a378ae60393a816",
        "trace_state": "[]"
    },
 

In [39]:
query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

{
    "name": "search_operation",
    "context": {
        "trace_id": "0xe9b307c9dad66e4f3d434e062bae578a",
        "span_id": "0x2e56a8c17eccfec9",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x2d5aae9eea97630f",
    "start_time": "2026-08-06T20:05:06.054659Z",
    "end_time": "2026-08-06T20:05:06.057288Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "ab10e358-cbfc-4b80-a29b-a2dbde7a07ea",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm_operation",
    "context": {
        "trace_id": "0xe9b307c9dad66e4f3d434e062bae578a",
        "span_id": "0xf41750398f3e0798",
        "trace_state": "[]"
    },
 

In [ ]:
"""
Then compute the input tokens for each llm span.

How much do the input tokens vary across these 4 runs?

They're identical
Within 10% of each other
Within 50% of each other
They vary more than 50%
"""

with sqlite3.connect("traces.db") as connection:
    df = pd.read_sql_query(
        """
        select * 
        from spans
        where name = 'llm_operation'
        """,
         connection

        )
    
df

# SOLUTION: the input tokes vary 0 percent among each other -> they are identical



,name,start_time,end_time,input_tokens,output_tokens,cost
0,llm_operation,1786046680897304174,1786046683147442704,7111,95,None
1,llm_operation,1786046697279320912,1786046699815060266,7111,94,None
2,llm_operation,1786046700971901003,1786046706032677151,7111,116,None
3,llm_operation,1786046706068281769,1786046707871402850,7111,93,None
